# Phase 5 – Feature Engineering v2 (erweitertes Feature-Set)

**Ziel:** `models/features_aggregated.pkl` (v1) laden und um Kontextdaten erweitern
(Feiertage/Ferien, Bevölkerung, Pflegeheime, Events, Unfallstatistik, Wetter) →
finaler 19-Feature-Vektor → `models/features_v2.pkl` speichern.

**Voraussetzung:** Phase 1 (`02_feature_engineering.ipynb`) muss abgeschlossen sein.

Regeln (AGENT.md):
- `models/features_aggregated.pkl` wird **nicht** überschrieben — Output ist `models/features_v2.pkl`
- Jeder Schritt wird mit einer Ausgabe verifiziert

In [1]:
import sys
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT.parent / 'Datensammlung'

from src.features import GEBIET_NAMES, GEBIET_COORDS, FEATURES_V2, map_plz_to_gebiet

print('Imports OK')

Imports OK


## 5.1 – Basis laden

In [2]:
PKL_PATH = ROOT / 'models' / 'features_aggregated.pkl'
df = joblib.load(PKL_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f'Geladen: {PKL_PATH}')
print(f'Shape: {df.shape}')
print(f'Zeitraum: {df["date"].min().date()} bis {df["date"].max().date()}')
df.head(3)

Geladen: C:\Users\lholz\OneDrive\Desktop\Diplomarbeit\RKA_RotKreuzApp\Modell\models\features_aggregated.pkl
Shape: (33437, 9)
Zeitraum: 2024-01-01 bis 2025-12-31


,date,gebiet_id,hour,weekday,month,quarter,is_weekend,season,einsatz_count
0,2024-01-01,1,0,0,1,1,0,0,1
1,2024-01-01,1,2,0,1,1,0,0,1
2,2024-01-01,1,8,0,1,1,0,0,1


## 5.2 – Feiertage & Ferien einbauen

In [3]:
with open(DATA_DIR / 'FeiertageFerien' / 'datamodeler.json', encoding='utf-8') as f:
    feiertage_raw = json.load(f)

feiertag_dates = set()
ferien_dates = set()
for entry in feiertage_raw:
    start = pd.to_datetime(entry['start'], format='%d.%m.%Y')
    ende = pd.to_datetime(entry['ende'], format='%d.%m.%Y')
    days = pd.date_range(start, ende, freq='D')
    if entry['typ'] == 'Feiertag':
        feiertag_dates.update(days)
    elif entry['typ'] == 'Ferien':
        ferien_dates.update(days)

df['is_holiday'] = df['date'].isin(feiertag_dates).astype(int)
df['is_school_holiday'] = df['date'].isin(ferien_dates).astype(int)

print(f'Feiertage geladen: {len(feiertag_dates)} Tage, Ferien geladen: {len(ferien_dates)} Tage')
print(f'Zeilen mit is_holiday=1: {df["is_holiday"].sum()}')
print(f'Zeilen mit is_school_holiday=1: {df["is_school_holiday"].sum()}')

Feiertage geladen: 60 Tage, Ferien geladen: 414 Tage
Zeilen mit is_holiday=1: 1628
Zeilen mit is_school_holiday=1: 9822


## 5.3 – Bevölkerungsdaten einbauen

In [4]:
with open(DATA_DIR / 'Bevoelkerungsdaten' / 'Bevoelkerungsdaten.json', encoding='utf-8') as f:
    bev_raw = json.load(f)

df_bev = pd.DataFrame(bev_raw)
df_bev['gebiet_id'] = df_bev['plz'].apply(map_plz_to_gebiet)
df_bev = df_bev[df_bev['gebiet_id'] != -1]

bev_agg = df_bev.groupby('gebiet_id').apply(lambda g: pd.Series({
    'population': g['gesBev'].sum(),
    'elderly_ratio': g['ueber65'].mean(),
    'elderly_abs': g['total_agegroup_4'].sum(),
}))

df['population'] = df['gebiet_id'].map(bev_agg['population'])
df['elderly_ratio'] = df['gebiet_id'].map(bev_agg['elderly_ratio'])
df['elderly_abs'] = df['gebiet_id'].map(bev_agg['elderly_abs'])

print('Statistik population / elderly_ratio / elderly_abs pro Gebiet:')
print(bev_agg.round(2))

Statistik population / elderly_ratio / elderly_abs pro Gebiet:
           population  elderly_ratio  elderly_abs
gebiet_id                                        
1            693794.0          20.70     180001.0
2            117684.0          18.21      30991.0
3            129786.0          19.44      37667.0
4            199669.0          19.61      39470.0
5            232826.0          18.53      46095.0
6            551613.0          18.68     146336.0
7            249026.0          19.49      72851.0
8             95394.0          22.10      29092.0
9            950488.0          19.00     133583.0
10            61202.0          25.31      20616.0
11           169552.0          18.70      46126.0
12           235369.0          18.64      64089.0
13           167644.0          19.66      32831.0
14           372553.0          19.12      62678.0
15           151318.0          19.48      43722.0


## 5.4 – Pflegeheime einbauen

In [5]:
with open(DATA_DIR / 'Pflegeheime' / 'Pflegeheime.json', encoding='utf-8') as f:
    pflege_raw = json.load(f)['pflegeheime']

df_pflege = pd.DataFrame(pflege_raw)
df_pflege['plz'] = df_pflege['plz'].astype(int)
df_pflege['gebiet_id'] = df_pflege['plz'].apply(map_plz_to_gebiet)
df_pflege = df_pflege[df_pflege['gebiet_id'] != -1]

beds_per_gebiet = df_pflege.groupby('gebiet_id')['pflegeplaetze'].sum()

df['nursing_home_beds'] = df['gebiet_id'].map(beds_per_gebiet).fillna(0).astype(int)

top5 = beds_per_gebiet.sort_values(ascending=False).head(5)
print('Top 5 Gebiete nach Pflegeplätzen:')
for gid, beds in top5.items():
    print(f'  {GEBIET_NAMES.get(gid, gid)}: {beds}')

n_ohne = (df.groupby('gebiet_id')['nursing_home_beds'].first() == 0).sum()
print(f'\nGebiete ohne Pflegeheim (nursing_home_beds=0): {n_ohne}')

Top 5 Gebiete nach Pflegeplätzen:
  Vöcklabruck + Gmunden Nord: 1952.0
  Linz: 1900.0
  Linz-Land: 1235.0
  Steyr: 1013.0
  Wels Stadt: 740.0

Gebiete ohne Pflegeheim (nursing_home_beds=0): 0


## 5.5 – Events einbauen

In [6]:
with open(DATA_DIR / 'Events' / 'Events.json', encoding='utf-8') as f:
    events_raw = json.load(f)['events']

event_pairs = set()
for ev in events_raw:
    gebiet_id = map_plz_to_gebiet(int(ev['plz']))
    if gebiet_id == -1:
        continue
    start = pd.to_datetime(ev['start_datum'])
    ende = pd.to_datetime(ev['end_datum'])
    for d in pd.date_range(start, ende, freq='D'):
        event_pairs.add((d, gebiet_id))

event_df = pd.DataFrame(list(event_pairs), columns=['date', 'gebiet_id'])
event_df['has_major_event'] = 1

df = df.merge(event_df, on=['date', 'gebiet_id'], how='left')
df['has_major_event'] = df['has_major_event'].fillna(0).astype(int)

print(f'Events geladen: {len(events_raw)}')
print(f'Einsatz-Stunden mit has_major_event=1: {df["has_major_event"].sum()}')

Events geladen: 331
Einsatz-Stunden mit has_major_event=1: 15404


## 5.6 – Unfallstatistik einbauen

In [7]:
df_unfall = pd.read_csv(
    DATA_DIR / 'Verkehrsdaten' / 'unfaelle_nach_gebiet_2024.csv',
    sep=';', decimal=',', encoding='cp1252'
)
df_unfall = df_unfall.rename(columns={'Gebiet_ID': 'gebiet_id'})
print(f'Spalten: {df_unfall.columns.tolist()}')

accident_rate_map = df_unfall.set_index('gebiet_id')['Rate_pro_10000_gewichtet']
df['accident_rate'] = df['gebiet_id'].map(accident_rate_map)

print(f'\naccident_rate Wertebereich: {df["accident_rate"].min()} - {df["accident_rate"].max()}')
print(f'Fehlende accident_rate: {df["accident_rate"].isna().sum()}')

Spalten: ['gebiet_id', 'Anzahl_PLZ', 'Bezirke_Basis', 'Rate_pro_10000_gewichtet', 'Bevoelkerung_geschaetzt', 'Unfaelle_gesamt_geschaetzt', 'Hotspot_Faktor', 'Unfaelle_hotspot_korrigiert', 'Verletzte_geschaetzt', 'Getoetete_geschaetzt']

accident_rate Wertebereich: 401 - 416
Fehlende accident_rate: 0


## 5.7 – Wetterdaten via Open-Meteo API laden

In [8]:
WEATHER_CACHE = ROOT / 'models' / 'weather_cache.pkl'

def fetch_weather_for_gebiet(gebiet_id, lat, lon):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': '2025-01-01', 'end_date': '2025-12-31',
        'hourly': 'temperature_2m,precipitation,snowfall,windspeed_10m',
        'timezone': 'Europe/Vienna',
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    h = r.json()['hourly']
    w = pd.DataFrame({
        'datetime': pd.to_datetime(h['time']),
        'temperature': h['temperature_2m'],
        'precipitation': h['precipitation'],
        'snowfall': h['snowfall'],
        'windspeed': h['windspeed_10m'],
    })
    w['gebiet_id'] = gebiet_id
    w['date'] = w['datetime'].dt.normalize()
    w['hour'] = w['datetime'].dt.hour
    return w

needed_dates = set(df['date'].unique())
needed_gebiete = set(df['gebiet_id'].unique())

coverage_ok = False
if WEATHER_CACHE.exists():
    weather_df = joblib.load(WEATHER_CACHE)
    weather_df['date'] = pd.to_datetime(weather_df['date'])
    coverage_ok = (
        needed_dates.issubset(set(weather_df['date'].unique()))
        and needed_gebiete.issubset(set(weather_df['gebiet_id'].unique()))
    )
    print(f'Wetter-Cache gefunden: {weather_df.shape}, deckt benoetigten Zeitraum/Gebiete ab: {coverage_ok}')
else:
    print('Kein Wetter-Cache gefunden.')

if not coverage_ok:
    frames = []
    for gid, (lat, lon) in GEBIET_COORDS.items():
        print(f'  Lade Wetterdaten Gebiet {gid} ({GEBIET_NAMES.get(gid, gid)})...')
        frames.append(fetch_weather_for_gebiet(gid, lat, lon))
        time.sleep(0.5)
    weather_df = pd.concat(frames, ignore_index=True)
    joblib.dump(weather_df, WEATHER_CACHE)
    print(f'Wetterdaten neu geladen und gecacht: {weather_df.shape}')

weather_join = weather_df[['gebiet_id', 'date', 'hour', 'temperature', 'precipitation', 'snowfall', 'windspeed']]
df = df.merge(weather_join, on=['gebiet_id', 'date', 'hour'], how='left')

n_missing = df['temperature'].isna().sum()
print(f'\nZeilen ohne Wetter-Match (werden mit Median gefuellt): {n_missing}')
for col in ['temperature', 'precipitation', 'snowfall', 'windspeed']:
    df[col] = df[col].fillna(df[col].median())

df['is_extreme_weather'] = (
    (df['temperature'] > 30) | (df['temperature'] < -5)
    | (df['snowfall'] > 2) | (df['windspeed'] > 50)
).astype(int)

print(f'is_extreme_weather=1: {df["is_extreme_weather"].sum()} von {len(df)} Zeilen')

Wetter-Cache gefunden: (263160, 9), deckt benoetigten Zeitraum/Gebiete ab: True

Zeilen ohne Wetter-Match (werden mit Median gefuellt): 0
is_extreme_weather=1: 612 von 33437 Zeilen


## 5.8 – Feature-Vektor finalisieren & speichern

In [9]:
print('NaN-Check vor Finalisierung:')
print(df[FEATURES_V2 + ['einsatz_count']].isnull().sum())

df_v2 = df[FEATURES_V2 + ['einsatz_count']].copy()

v1_shape = joblib.load(ROOT / 'models' / 'features_aggregated.pkl').shape
print(f'\nv1 Shape: {v1_shape}  ({v1_shape[1] - 1} Features + Ziel)')
print(f'v2 Shape: {df_v2.shape}  ({df_v2.shape[1] - 1} Features + Ziel)')

out_path = ROOT / 'models' / 'features_v2.pkl'
joblib.dump(df_v2, out_path)
print(f'\nGespeichert: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)')
df_v2.head(3)

NaN-Check vor Finalisierung:
gebiet_id             0
hour                  0
weekday               0
month                 0
quarter               0
is_weekend            0
season                0
is_holiday            0
is_school_holiday     0
population            0
elderly_ratio         0
nursing_home_beds     0
has_major_event       0
accident_rate         0
temperature           0
precipitation         0
snowfall              0
windspeed             0
is_extreme_weather    0
einsatz_count         0
dtype: int64

v1 Shape: (33437, 9)  (8 Features + Ziel)
v2 Shape: (33437, 20)  (19 Features + Ziel)

Gespeichert: C:\Users\lholz\OneDrive\Desktop\Diplomarbeit\RKA_RotKreuzApp\Modell\models\features_v2.pkl (4703.7 KB)


,gebiet_id,hour,weekday,month,quarter,is_weekend,season,is_holiday,is_school_holiday,population,elderly_ratio,nursing_home_beds,has_major_event,accident_rate,temperature,precipitation,snowfall,windspeed,is_extreme_weather,einsatz_count
0,1,0,0,1,1,0,0,1,1,693794.0,20.7,1900,0,401,3.5,0.6,0.0,10.9,0,1
1,1,2,0,1,1,0,0,1,1,693794.0,20.7,1900,0,401,4.2,0.8,0.0,13.0,0,1
2,1,8,0,1,1,0,0,1,1,693794.0,20.7,1900,0,401,3.2,0.0,0.0,7.2,0,1


---
## ✅ Phase 5 abgeschlossen
- `models/features_v2.pkl` existiert, 19 Features ohne NaN
- Shape größer als v1 (zusätzliche Spalten: Feiertage, Bevölkerung, Pflegeheime, Events, Unfallstatistik, Wetter)